In [1]:
import pandas as pd  # pour manipuler les tableaux de données (dataframes)
import requests  # pour aller chercher les données sur Sofascore et Understat
from unidecode import unidecode  # pour normaliser les noms de joueurs (enlever les accents)
import re  # pour nettoyer le texte si besoin (tirets, apostrophes)

In [2]:
%pip install unidecode

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install cloudscraper 

Note: you may need to restart the kernel to use updated packages.


In [9]:
import cloudscraper  # on importe la librairie qui va remplacer requests pour ce site précis

scraper = cloudscraper.create_scraper()  # crée un "scraper" qui se comporte comme un vrai navigateur face aux protections anti-bot, contrairement à requests.Session()

url = "https://api.sofascore.com/api/v1/search/all?q=Serie%20A&page=0"  # toujours le même endpoint de recherche

response = scraper.get(url)  # on envoie la requête avec le scraper, pas besoin de headers manuels car cloudscraper les génère lui-même de façon réaliste

print(response.status_code)  # on vérifie si le code passe à 200 cette fois

200


In [10]:
data = response.json()  # on transforme la réponse en dictionnaire Python, plus facile à explorer que du texte brut

print(data.keys())  # on affiche d'abord les clés principales du JSON, pour comprendre comment la réponse est structurée avant de creuser dedans

dict_keys(['results'])


In [11]:
print(len(data["results"]))  # on affiche le nombre total de résultats, pour savoir si la recherche a été large ou non

print(data["results"][0])  # on affiche le tout premier résultat en entier, pour voir sa structure (les clés qu'il contient) avant de chercher celui qui correspond à la Serie A

20
{'entity': {'id': 23, 'name': 'Serie A', 'slug': 'serie-a', 'userCount': 712104, 'category': {'id': 31, 'name': 'Italy', 'slug': 'italy', 'alpha2': 'IT', 'flag': 'italy', 'sport': {'id': 1, 'slug': 'football', 'name': 'Football'}, 'country': {'alpha2': 'IT', 'name': 'Italy', 'slug': 'italy'}}, 'displayInverseHomeAwayTeams': False, 'fieldTranslations': {'nameTranslation': {'ar': 'الدوري الإيطالي', 'bn': 'সিরি আ', 'hi': 'सीरी ए', 'ru': 'Серия А'}, 'shortNameTranslation': {}}, 'gender': 'M'}, 'score': 101684.56, 'type': 'uniqueTournament'}


In [12]:
serie_a = None  # variable qui contiendra le bon résultat une fois trouvé, None tant qu'on ne l'a pas identifié

for item in data["results"]:  # on parcourt chaque résultat de la recherche un par un
    entity = item["entity"]  # on extrait la partie "entity" qui contient les infos du tournoi/équipe/joueur
    if item["type"] == "uniqueTournament" and entity["name"] == "Serie A" and entity["category"]["name"] == "Italy":  # on vérifie les trois critères en même temps pour être sûr de ne pas se tromper de compétition
        serie_a = entity  # on garde ce résultat car il correspond bien à la Serie A italienne
        break  # on arrête la boucle, plus besoin de continuer une fois trouvé

print(serie_a["id"])  # on affiche l'ID final, celui qu'on utilisera pour toutes les requêtes suivantes (classement, joueurs, stats...)

23


In [13]:
url_seasons = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/seasons"  # endpoint qui liste toutes les saisons disponibles pour ce tournoi précis (Serie A, id 23)

response_seasons = scraper.get(url_seasons)  # on réutilise le même scraper que précédemment, il garde les protections déjà résolues

print(response_seasons.status_code)  # on vérifie d'abord que la requête passe bien avant de creuser dans les données

data_seasons = response_seasons.json()  # on transforme la réponse en dictionnaire Python

print(data_seasons.keys())  # on regarde la structure du JSON, comme pour la première requête, avant de chercher la saison qui nous intéresse

200
dict_keys(['seasons'])


In [14]:
print(len(data_seasons["seasons"]))  # on affiche le nombre total de saisons disponibles, pour voir jusqu'où remonte l'historique

print(data_seasons["seasons"][0])  # on affiche la première saison de la liste, elle est probablement triée de la plus récente à la plus ancienne

62
{'name': 'Serie A 26/27', 'year': '26/27', 'editor': False, 'id': 95836}


In [16]:
season = None  # variable qui contiendra la bonne saison une fois trouvée, None tant qu'on ne l'a pas identifiée

for item in data_seasons["seasons"]:  # on parcourt chaque saison de la liste une par une
    if item["year"] == "25/26":  # on cherche maintenant la saison 2025/2026, la saison précédente (déjà terminée), plutôt que la saison en cours
        season = item  # on garde cette saison car elle correspond à celle qu'on veut
        break  # on arrête la boucle dès qu'on l'a trouvée, inutile de continuer

print(season["id"])  # on affiche l'ID final de la saison 25/26, celui qu'on utilisera pour récupérer le classement et les joueurs de cette saison précise

76457


In [17]:
url_stats = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics"  # endpoint de statistiques agrégées par joueur, pour la Serie A saison 25/26

params = {
    "limit": 20,  # nombre de joueurs à récupérer par page, on teste petit d'abord pour voir la structure
    "order": "-rating",  # on trie par note décroissante, juste pour avoir un aperçu, on changera le tri plus tard
    "type": "overall",  # type de statistiques : overall = vue d'ensemble, il existe aussi "attack", "defence", "passing" qu'on utilisera ensuite
}  # dictionnaire des paramètres de la requête, séparé de l'URL pour plus de lisibilité

response_stats = scraper.get(url_stats, params=params)  # on envoie la requête avec ces paramètres

print(response_stats.status_code)  # on vérifie d'abord que ça passe

data_stats = response_stats.json()  # on transforme la réponse en dictionnaire Python

print(data_stats.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['results', 'page', 'pages'])


In [18]:
print(data_stats["pages"])  # on affiche le nombre total de pages, pour savoir combien de requêtes il faudra faire pour tout récupérer

print(data_stats["results"][0])  # on affiche le premier joueur en entier, pour voir toutes les statistiques disponibles avec le type "overall"

30
{'player': {'name': 'Oliver Christensen', 'slug': 'oliver-christensen', 'userCount': 639, 'gender': 'M', 'id': 860306, 'fieldTranslations': {'nameTranslation': {'ar': 'أوليفير كريستنسن', 'bn': 'অলিভার ক্রিস্টেনসেন', 'hi': 'ओलिवर क्रिस्टेंसन', 'ru': 'Оливер Кристенсен'}, 'shortNameTranslation': {'ar': 'أ. كريستنسن', 'bn': 'ও. ক্রিস্টেনসেন', 'hi': 'ओ. क्रिस्टेंसन', 'ru': 'О. Кристенсен'}}}, 'team': {'name': 'Fiorentina', 'slug': 'fiorentina', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'userCount': 0, 'national': False, 'type': 0, 'id': 2693, 'teamColors': {'primary': '#374df5', 'secondary': '#374df5', 'text': '#ffffff'}, 'fieldTranslations': {'nameTranslation': {'ar': 'فيورنتينا', 'bn': 'ফিওরেন্টিনা', 'hi': 'फियोरेंटीना', 'ru': 'Фиорентина'}, 'shortNameTranslation': {'ar': 'فيورنتينا'}}}}


In [19]:
url_top = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/top-players/overall"  # endpoint "top players" qui devrait contenir la valeur de chaque statistique, contrairement à l'endpoint précédent

response_top = scraper.get(url_top)  # requête avec le scraper cloudscraper, toujours nécessaire face à la protection Cloudflare

print(response_top.status_code)  # on vérifie d'abord que ça passe

data_top = response_top.json()  # on transforme la réponse en dictionnaire Python

print(data_top.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['topPlayers', 'statisticsType'])


In [20]:
print(data_top["statisticsType"])  # on affiche le type de statistiques renvoyé, pour confirmer qu'on est bien sur une vue "overall"

print(data_top["topPlayers"].keys())  # topPlayers est probablement un dictionnaire avec une clé par catégorie de stat (buts, passes, notes...), on regarde lesquelles sont disponibles

{'sportSlug': 'football', 'statisticsType': 'player'}
dict_keys(['rating', 'goals', 'expectedGoals', 'assists', 'expectedAssists', 'goalsAssistsSum', 'kilometersCoveredPer90', 'numberOfSprintsPer90', 'topSpeed', 'penaltyGoals', 'freeKickGoal', 'scoringFrequency', 'totalShots', 'shotsOnTarget', 'bigChancesMissed', 'bigChancesCreated', 'accuratePasses', 'keyPasses', 'accurateLongBalls', 'successfulDribbles', 'penaltyWon', 'tackles', 'interceptions', 'clearances', 'possessionLost', 'yellowCards', 'redCards', 'saves', 'goalsPrevented', 'mostConceded', 'leastConceded', 'cleanSheet'])


In [21]:
print(len(data_top["topPlayers"]["goals"]))  # on compte combien de joueurs sont listés dans la catégorie "goals", pour voir si c'est limité ou si ça couvre tout le championnat

print(data_top["topPlayers"]["goals"][0])  # on regarde le premier élément de la catégorie "goals", pour voir sa structure exacte (nom du joueur, équipe, valeur de la stat...)

50
{'statistics': {'goals': 17, 'id': 2176240, 'type': 'overall', 'appearances': 30, 'statisticsType': {'sportSlug': 'football', 'statisticsType': 'player'}}, 'playedEnough': True, 'player': {'name': 'Lautaro Martínez', 'slug': 'lautaro-martinez', 'shortName': 'L. Martínez', 'position': 'F', 'userCount': 143531, 'gender': 'M', 'id': 823984, 'fieldTranslations': {'nameTranslation': {'ar': 'لاوتارو مارتينيز', 'bn': 'লাউতারো মার্টিনেজ', 'hi': 'लौटारो मार्टिनेज़', 'ru': 'Лаутаро Мартинес'}, 'shortNameTranslation': {'ar': 'ل. مارتينيز', 'bn': 'এল. মার্টিনেজ', 'hi': 'एल. मार्टिनेज़', 'ru': 'Л. Мартинес'}}}, 'team': {'name': 'Inter', 'slug': 'inter', 'gender': 'M', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'userCount': 2114574, 'national': False, 'type': 0, 'id': 2697, 'teamColors': {'primary': '#374df5', 'secondary': '#374df5', 'text': '#ffffff'}, 'fieldTranslations': {'nameTranslation': {'ar': 'انتر', 'bn': 'ইন্টার', 'hi': 'इंटर', 'ru': 'Интер'}, 'shortNameTranslation': {}}}

In [22]:
url_standings = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/standings/total"  # endpoint du classement complet, qui liste forcément toutes les équipes de la saison

response_standings = scraper.get(url_standings)  # requête avec le scraper, toujours nécessaire

print(response_standings.status_code)  # on vérifie que ça passe

data_standings = response_standings.json()  # transforme la réponse en dictionnaire Python

print(data_standings.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['standings'])


In [23]:
print(len(data_standings["standings"]))  # on affiche le nombre d'éléments, au cas où il y aurait plusieurs groupes (rare en Serie A, mais on vérifie)

print(data_standings["standings"][0].keys())  # on regarde les clés du premier élément, pour repérer où se trouve la liste des équipes

1
dict_keys(['id', 'type', 'tournament', 'name', 'descriptions', 'tieBreakingRule', 'rows', 'updatedAtTimestamp'])


In [24]:
print(len(data_standings["standings"][0]["rows"]))  # on compte le nombre d'équipes dans le classement, devrait être 20 pour la Serie A

print(data_standings["standings"][0]["rows"][0])  # on affiche la première équipe du classement en entier, pour voir où se trouve son id

20
{'id': 1609091, 'team': {'name': 'Inter', 'slug': 'inter', 'shortName': 'Inter', 'gender': 'M', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'userCount': 2114574, 'nameCode': 'INT', 'disabled': False, 'national': False, 'type': 0, 'country': {'alpha2': 'IT', 'alpha3': 'ITA', 'name': 'Italy', 'slug': 'italy'}, 'id': 2697, 'teamColors': {'primary': '#1a57cc', 'secondary': '#000000', 'text': '#000000'}, 'fieldTranslations': {'nameTranslation': {'ar': 'انتر', 'bn': 'ইন্টার', 'hi': 'इंटर', 'ru': 'Интер'}, 'shortNameTranslation': {}}}, 'descriptions': [], 'promotion': {'id': 804, 'text': 'Champions League'}, 'position': 1, 'matches': 38, 'wins': 27, 'losses': 5, 'draws': 6, 'scoresFor': 89, 'scoresAgainst': 35, 'points': 87, 'scoreDiffFormatted': '+54'}


In [25]:
teams = []  # liste qui contiendra un dictionnaire par équipe (nom + id), pour pouvoir boucler dessus ensuite

for row in data_standings["standings"][0]["rows"]:  # on parcourt chaque ligne du classement, une par équipe
    teams.append({"name": row["team"]["name"], "id": row["team"]["id"]})  # on garde uniquement le nom et l'id, les seules infos utiles pour la suite

print(len(teams))  # on vérifie qu'on a bien 20 équipes

print(teams[0])  # on affiche la première pour vérifier le format

20
{'name': 'Inter', 'id': 2697}


In [26]:
team_test_id = teams[0]["id"]  # on prend l'id de la première équipe (Inter) pour faire le test avant de généraliser à toutes

url_team_stats = f"https://api.sofascore.com/api/v1/team/{team_test_id}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # endpoint stats spécifique à une équipe, censé contenir tous ses joueurs contrairement au top-players limité à 50

response_team_stats = scraper.get(url_team_stats)  # requête avec le scraper

print(response_team_stats.status_code)  # on vérifie que ça passe

data_team_stats = response_team_stats.json()  # transforme en dictionnaire Python

print(data_team_stats.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['statistics'])


In [28]:
print(list(data_team_stats["statistics"].keys())[:15])  # on affiche les 15 premières clés du dictionnaire, pour voir si ce sont des noms de stats (agrégées équipe) ou des ids de joueurs

print(data_team_stats["statistics"]["goals"] if "goals" in data_team_stats["statistics"] else "pas de clé 'goals' directe")  # test rapide : si "goals" existe en clé directe, ça confirme que c'est agrégé au niveau équipe, pas par joueur

['goalsScored', 'goalsConceded', 'ownGoals', 'assists', 'shots', 'penaltyGoals', 'penaltiesTaken', 'freeKickGoals', 'freeKickShots', 'goalsFromInsideTheBox', 'goalsFromOutsideTheBox', 'shotsFromInsideTheBox', 'shotsFromOutsideTheBox', 'headedGoals', 'leftFootGoals']
pas de clé 'goals' directe


In [29]:
url_roster = f"https://api.sofascore.com/api/v1/team/{team_test_id}/players"  # endpoint qui liste l'effectif complet d'une équipe

response_roster = scraper.get(url_roster)  # requête avec le scraper

print(response_roster.status_code)  # on vérifie que ça passe

data_roster = response_roster.json()  # transforme en dictionnaire Python

print(data_roster.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['players', 'foreignPlayers', 'nationalPlayers', 'supportStaff', 'playerPreviousTeam', 'nationalTeamPlayerStatistics', 'teamDepthAssignments'])


In [30]:
print(len(data_roster["players"]))  # on compte le nombre de joueurs dans l'effectif

print(data_roster["players"][0])  # on affiche le premier joueur pour voir sa structure, notamment où se trouve son id

25
{'player': {'name': 'Lautaro Martínez', 'firstName': 'Lautaro', 'lastName': 'Martinez', 'slug': 'lautaro-martinez', 'shortName': 'L. Martínez', 'team': {'name': 'Inter', 'slug': 'inter', 'shortName': 'Inter', 'gender': 'M', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'tournament': {'name': 'Serie A', 'slug': 'serie-a', 'category': {'name': 'Italy', 'slug': 'italy', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'priority': 7, 'country': {'alpha2': 'IT', 'alpha3': 'ITA', 'name': 'Italy', 'slug': 'italy'}, 'id': 31, 'flag': 'italy', 'alpha2': 'IT', 'fieldTranslations': {'nameTranslation': {'ar': 'إيطاليا', 'bn': 'ইতালি', 'hi': 'इटली', 'ru': 'Италия'}, 'shortNameTranslation': {}}}, 'uniqueTournament': {'name': 'Serie A', 'slug': 'serie-a', 'primaryColorHex': '#09519e', 'secondaryColorHex': '#008fd7', 'category': {'name': 'Italy', 'slug': 'italy', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'priority': 7, 'country': {'alpha2': 'IT', 'alpha3':

In [31]:
roster_ids = [p["player"]["id"] for p in data_roster["players"]]  # on extrait uniquement les id des 25 joueurs de l'effectif, dans une liste simple

print(roster_ids[:5])  # on affiche les 5 premiers pour vérifier que ce sont bien des nombres (les id joueurs) et pas autre chose

url_player_stats = f"https://api.sofascore.com/api/v1/player/{roster_ids[0]}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # endpoint stats individuel, testé sur le premier joueur de la liste (Lautaro Martínez)

response_player_stats = scraper.get(url_player_stats)  # requête avec le scraper

print(response_player_stats.status_code)  # on vérifie que ça passe

data_player_stats = response_player_stats.json()  # transforme en dictionnaire Python

print(data_player_stats)  # on affiche tout pour voir la structure complète cette fois, vu qu'on est sur un seul joueur

[823984, 791702, 1086223, 1156616, 1530727]
200
{'statistics': {'rating': 7.09, 'totalRating': 212.7, 'countRating': 30, 'goals': 17, 'bigChancesCreated': 12, 'bigChancesMissed': 16, 'assists': 6, 'expectedAssists': 2.70818706, 'goalsAssistsSum': 23, 'expectedGoalsInvolvement': 16.20378706, 'accuratePasses': 469, 'inaccuratePasses': 131, 'totalPasses': 600, 'accuratePassesPercentage': 78.166666666667, 'accurateOwnHalfPasses': 133, 'accurateOppositionHalfPasses': 336, 'accurateFinalThirdPasses': 217, 'keyPasses': 37, 'successfulDribbles': 21, 'successfulDribblesPercentage': 47.727272727273, 'tackles': 24, 'interceptions': 7, 'yellowCards': 4, 'directRedCards': 0, 'redCards': 0, 'accurateCrosses': 2, 'accurateCrossesPercentage': 11.764705882353, 'totalShots': 92, 'shotsOnTarget': 39, 'shotsOffTarget': 30, 'groundDuelsWon': 91, 'groundDuelsWonPercentage': 45.959595959596, 'aerialDuelsWon': 24, 'aerialDuelsWonPercentage': 42.857142857143, 'totalDuelsWon': 115, 'totalDuelsWonPercentage': 45

In [33]:
colonnes_identite = [
    "id",  # id sofascore du joueur, indispensable pour le matching avec Understat plus tard
    "name",  # nom complet du joueur
    "position",  # poste en une lettre (G, D, M, F), pour le regroupement large défense/milieu/attaque
    "dateOfBirth",  # utile pour vérifier qu'on ne confond pas deux joueurs homonymes lors du matching
]

colonnes_temps_de_jeu = [
    "minutesPlayed",  # sert à appliquer le seuil de 900 minutes minimum déjà validé
    "appearances",  # nombre de matchs joués
    "matchesStarted",  # nombre de titularisations, utile pour distinguer un titulaire d'un joker de luxe
]

colonnes_offensives = [
    "goals",  # buts marqués
    "expectedGoals",  # xG, équivalent de ce qu'on récupère sur Understat
    "assists",  # passes décisives
    "expectedAssists",  # xA, équivalent Understat aussi
    "bigChancesCreated",  # grosses occasions créées pour un coéquipier
    "totalShots",  # nombre de tirs total, utile pour juger le volume de jeu offensif
    "shotsOnTarget",  # tirs cadrés, complète totalShots
    "successfulDribbles",  # dribbles réussis, utile pour juger le profil technique
    "keyPasses",  # passes clés (menant à un tir), proche de bigChancesCreated mais plus large
]

colonnes_passes_possession = [
    "accuratePasses",  # passes réussies, volume brut
    "totalPasses",  # passes tentées, pour calculer le pourcentage si besoin
    "accuratePassesPercentage",  # taux de réussite des passes, utile pour juger la fiabilité technique
    "accurateLongBalls",  # longs ballons réussis, utile pour les relanceurs (défenseurs, milieux)
    "touches",  # nombre de touches de balle, indicateur d'implication dans le jeu
]

colonnes_defensives = [
    "tackles",  # tacles tentés
    "tacklesWon",  # tacles réussis, plus pertinent que le total seul
    "interceptions",  # interceptions
    "clearances",  # dégagements
    "blockedShots",  # tirs contrés
    "aerialDuelsWon",  # duels aériens gagnés
    "aerialDuelsWonPercentage",  # taux de réussite dans les airs, important pour les défenseurs centraux
    "groundDuelsWon",  # duels au sol gagnés
]

colonnes_discipline = [
    "yellowCards",  # cartons jaunes
    "redCards",  # cartons rouges
    "fouls",  # fautes commises
]

colonnes_gardien = [
    "saves",  # arrêts, uniquement pertinent pour les gardiens (position == 'G')
    "cleanSheet",  # matchs sans encaisser de but
    "goalsConceded",  # buts encaissés
    "penaltySave",  # penalties arrêtés
]

In [34]:
colonnes_a_garder = (
    colonnes_identite
    + colonnes_temps_de_jeu
    + colonnes_offensives
    + colonnes_passes_possession
    + colonnes_defensives
    + colonnes_discipline
    + colonnes_gardien
)  # on fusionne toutes les catégories validées plus haut en une seule liste, pour filtrer en une seule passe

def extraire_stats_joueur(player_info, stats_json):
    # player_info : dictionnaire du roster (id, nom, poste, date de naissance, équipe)
    # stats_json : dictionnaire complet renvoyé par l'endpoint statistics/overall pour ce joueur
    stats = stats_json.get("statistics", {})  # sous-dictionnaire des stats, vide par défaut pour éviter un crash si la clé manque

    ligne = {}  # dictionnaire qui contiendra une seule ligne de notre future table (un joueur)

    ligne["id"] = player_info["id"]  # id sofascore, clé de matching future avec Understat
    ligne["name"] = player_info["name"]  # nom complet du joueur
    ligne["position"] = player_info["position"]  # poste en une lettre (G, D, M, F)
    ligne["dateOfBirth"] = player_info.get("dateOfBirth")  # utile pour désambiguïser les homonymes lors du matching
    ligne["team"] = player_info["team"]["name"]  # nom de l'équipe, pris depuis le roster pour être sûr de la bonne saison

    for colonne in colonnes_a_garder:  # on parcourt toutes les colonnes de stats choisies
        if colonne not in ligne:  # on évite d'écraser les colonnes d'identité déjà ajoutées juste au-dessus
            ligne[colonne] = stats.get(colonne)  # .get() renvoie None si la clé n'existe pas (ex: stats gardien chez un attaquant), plutôt que de planter

    return ligne  # dictionnaire complet pour ce joueur, prêt à être ajouté à la liste finale

In [35]:
import time  # pour ajouter la pause de 2 secondes entre chaque requête

toutes_les_lignes = []  # liste qui contiendra un dictionnaire par joueur, pour toute la Serie A

for equipe in teams:  # on parcourt les 20 équipes récupérées depuis le classement
    url_roster = f"https://api.sofascore.com/api/v1/team/{equipe['id']}/players"  # effectif complet de cette équipe

    response_roster = scraper.get(url_roster)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes après cette requête, pour ne pas surcharger l'API

    if response_roster.status_code != 200:  # sécurité : si le roster échoue, on log et on passe à l'équipe suivante sans faire planter tout le script
        print(f"Erreur roster pour {equipe['name']} : {response_roster.status_code}")
        continue  # équipe suivante

    data_roster = response_roster.json()  # transforme la réponse en dictionnaire Python

    for joueur_roster in data_roster["players"]:  # on parcourt chaque joueur de l'effectif
        player_info = joueur_roster["player"]  # sous-dictionnaire des infos du joueur

        url_stats = f"https://api.sofascore.com/api/v1/player/{player_info['id']}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # stats individuelles du joueur

        response_stats = scraper.get(url_stats)  # requête avec le scraper
        time.sleep(2)  # pause de 2 secondes après cette requête aussi

        if response_stats.status_code != 200:  # sécurité : certains joueurs (jamais entrés en jeu, blessés toute la saison) peuvent ne pas avoir de stats
            print(f"Pas de stats pour {player_info['name']} ({equipe['name']}) : {response_stats.status_code}")
            continue  # joueur suivant

        stats_json = response_stats.json()  # transforme la réponse en dictionnaire Python

        ligne = extraire_stats_joueur(player_info, stats_json)  # on filtre uniquement les colonnes choisies

        toutes_les_lignes.append(ligne)  # on ajoute ce joueur à la liste finale

    print(f"{equipe['name']} terminé, {len(data_roster['players'])} joueurs traités")  # suivi de la progression équipe par équipe

print(f"Total : {len(toutes_les_lignes)} joueurs récupérés")  # bilan final

Pas de stats pour Curtis Jones (Inter) : 404
Pas de stats pour Aleksandar Stanković (Inter) : 404
Pas de stats pour John Stones (Inter) : 404
Pas de stats pour Djed Spence (Inter) : 404
Inter terminé, 25 joueurs traités
Pas de stats pour Costantino Favasuli (SSC Napoli) : 404
Pas de stats pour Benoît Badiashile (SSC Napoli) : 404
Pas de stats pour Rafa Marín (SSC Napoli) : 404
Pas de stats pour Claudio Pugliese (SSC Napoli) : 404
SSC Napoli terminé, 27 joueurs traités
Pas de stats pour Rodrigo Mora (AS Roma) : 404
Pas de stats pour Emanuele Lulli (AS Roma) : 404
Pas de stats pour Nahuel Molina (AS Roma) : 404
Pas de stats pour Leonardo Balerdi (AS Roma) : 404
Pas de stats pour Konstantinos Koulierakis (AS Roma) : 404
Pas de stats pour Devis Vásquez (AS Roma) : 404
Pas de stats pour Pierluigi Gollini (AS Roma) : 404
Pas de stats pour Giorgio De Marzi (AS Roma) : 404
AS Roma terminé, 25 joueurs traités
Pas de stats pour Mattia Liberali (Como) : 404
Pas de stats pour Yan Couto (Como) : 40

In [36]:
import pandas as pd  # librairie pour manipuler les données sous forme de table

df = pd.DataFrame(toutes_les_lignes)  # on transforme la liste de dictionnaires en DataFrame, une ligne par joueur

print(df.shape)  # on affiche (nombre de lignes, nombre de colonnes), pour vérifier qu'on a bien 387 lignes et le bon nombre de colonnes

print(df.head())  # on affiche les 5 premières lignes pour un premier coup d'oeil visuel

(387, 37)
        id              name position                dateOfBirth   team  \
0   823984  Lautaro Martínez        F  1997-08-22T00:00:00+00:00  Inter   
1   791702     Marcus Thuram        F  1997-08-06T00:00:00+00:00  Inter   
2  1086223   Ange-Yoan Bonny        F  2003-10-25T00:00:00+00:00  Inter   
3  1156616      Pio Esposito        F  2005-06-28T00:00:00+00:00  Inter   
4  1530727    Mattia Mosconi        F  2007-03-26T00:00:00+00:00  Inter   

   minutesPlayed  appearances  matchesStarted  goals  expectedGoals  ...  \
0           2178           30              27     17        13.4956  ...   
1           1913           29              24     13         9.4339  ...   
2           1155           33              10      5         3.7935  ...   
3           1572           35              15      7         7.3965  ...   
4             25            2               0      0         0.0943  ...   

   aerialDuelsWon  aerialDuelsWonPercentage  groundDuelsWon  yellowCards  \
0     

In [37]:
url_top_test = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/top-players/overall"  # même endpoint que précédemment

params_test = {"limit": 600}  # on tente un plafond largement supérieur au nombre de joueurs de la Serie A, pour voir si l'API l'accepte ou l'ignore

response_top_test = scraper.get(url_top_test, params=params_test)  # requête avec le paramètre limit en plus

print(response_top_test.status_code)  # on vérifie que ça passe

data_top_test = response_top_test.json()  # transforme en dictionnaire Python

print(len(data_top_test["topPlayers"]["goals"]))  # on compare à 50 : si c'est plus grand, le paramètre limit fonctionne

200
50


In [38]:
url_events = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/events/round/1"  # endpoint qui liste les matchs de la journée 1 de la saison 25/26

response_events = scraper.get(url_events)  # requête avec le scraper

print(response_events.status_code)  # on vérifie que ça passe

data_events = response_events.json()  # transforme en dictionnaire Python

print(data_events.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['events', 'hasNextPage'])


In [39]:
print(len(data_events["events"]))  # on compte le nombre de matchs pour cette journée, devrait être 10 (20 équipes / 2)

premier_match = data_events["events"][0]  # on prend le premier match de la journée pour le test

print(premier_match["homeTeam"]["name"], "vs", premier_match["awayTeam"]["name"])  # affiche les deux équipes, pour identifier le match

print(premier_match["id"])  # affiche l'id du match, celui qu'on va utiliser pour interroger /lineups

10
Genoa vs Lecce
13981422


In [40]:
url_lineups = f"https://api.sofascore.com/api/v1/event/13981422/lineups"  # endpoint des compositions pour ce match précis

response_lineups = scraper.get(url_lineups)  # requête avec le scraper

print(response_lineups.status_code)  # on vérifie que ça passe

data_lineups = response_lineups.json()  # transforme en dictionnaire Python

print(data_lineups.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['confirmed', 'home', 'away', 'statisticalVersion'])


In [41]:
print(data_lineups["home"].keys())  # on regarde les clés côté équipe à domicile, pour repérer où se trouve la liste des joueurs

print(len(data_lineups["home"]["players"]))  # on compte le nombre de joueurs listés côté domicile, devrait inclure titulaires + remplaçants

print(data_lineups["home"]["players"][0])  # on affiche le premier joueur pour voir sa structure exacte, notamment comment on distingue titulaire/remplaçant

dict_keys(['players', 'supportStaff', 'formation', 'playerColor', 'goalkeeperColor', 'missingPlayers'])
24
{'player': {'name': 'Nicola Leali', 'slug': 'nicola-leali', 'shortName': 'N. Leali', 'position': 'G', 'jerseyNumber': '1', 'height': 193, 'userCount': 482, 'gender': 'M', 'sofascoreId': 'vxb-uzs', 'country': {'alpha2': 'IT', 'alpha3': 'ITA', 'name': 'Italy', 'slug': 'italy'}, 'id': 98059, 'marketValueCurrency': 'EUR', 'dateOfBirthTimestamp': 729907200, 'proposedMarketValueRaw': {'value': 920000, 'currency': 'EUR'}, 'fieldTranslations': {'nameTranslation': {'ar': 'نيكولا ليالي', 'bn': 'নিকোলা লেইলি', 'hi': 'निकोला लीली', 'ru': 'Никола Леали'}, 'shortNameTranslation': {'ar': 'ن. ليالي', 'bn': 'এন. লেইলি', 'hi': 'एन. लीली', 'ru': 'Н. Леали'}}}, 'teamId': 2701, 'shirtNumber': 1, 'jerseyNumber': '1', 'position': 'G', 'substitute': False, 'statistics': {'totalPass': 33, 'accuratePass': 14, 'totalLongBalls': 20, 'accurateLongBalls': 2, 'goalAssist': 0, 'accurateOwnHalfPasses': 13, 'total

In [42]:
ids_joueurs_saison = set()  # ensemble (pas une liste) pour stocker tous les id de joueurs sans doublons automatiquement

for journee in range(1, 39):  # on parcourt les 38 journées de la Serie A (round 1 à 38 inclus)
    url_events = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/events/round/{journee}"  # matchs de cette journée précise

    response_events = scraper.get(url_events)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes, comme prévu

    if response_events.status_code != 200:  # sécurité : si une journée échoue, on log et on continue plutôt que de tout arrêter
        print(f"Erreur journée {journee} : {response_events.status_code}")
        continue  # journée suivante

    data_events = response_events.json()  # transforme en dictionnaire Python

    for match in data_events["events"]:  # on parcourt chaque match de cette journée
        url_lineups = f"https://api.sofascore.com/api/v1/event/{match['id']}/lineups"  # compositions de ce match précis

        response_lineups = scraper.get(url_lineups)  # requête avec le scraper
        time.sleep(2)  # pause de 2 secondes après chaque match aussi

        if response_lineups.status_code != 200:  # sécurité : certains matchs peuvent ne pas avoir de lineups enregistrées (report, annulation)
            print(f"Pas de lineups pour le match {match['id']} : {response_lineups.status_code}")
            continue  # match suivant

        data_lineups = response_lineups.json()  # transforme en dictionnaire Python

        for cote in ["home", "away"]:  # on traite les deux équipes du match, domicile puis extérieur
            if cote not in data_lineups or "players" not in data_lineups[cote]:  # sécurité : certains matchs très récents ou reportés peuvent manquer cette clé
                continue  # équipe suivante

            for joueur in data_lineups[cote]["players"]:  # on parcourt chaque joueur listé (titulaire + remplaçant confondus)
                ids_joueurs_saison.add(joueur["player"]["id"])  # ajout à l'ensemble, les doublons sont ignorés automatiquement

    print(f"Journée {journee} terminée, {len(ids_joueurs_saison)} id uniques cumulés jusqu'ici")  # suivi de la progression journée par journée

print(f"Total final : {len(ids_joueurs_saison)} joueurs uniques ayant figuré dans une feuille de match en 25/26")  # bilan final

Journée 1 terminée, 474 id uniques cumulés jusqu'ici
Journée 2 terminée, 514 id uniques cumulés jusqu'ici
Journée 3 terminée, 564 id uniques cumulés jusqu'ici
Journée 4 terminée, 579 id uniques cumulés jusqu'ici
Journée 5 terminée, 584 id uniques cumulés jusqu'ici
Journée 6 terminée, 589 id uniques cumulés jusqu'ici
Journée 7 terminée, 598 id uniques cumulés jusqu'ici
Journée 8 terminée, 604 id uniques cumulés jusqu'ici
Journée 9 terminée, 612 id uniques cumulés jusqu'ici
Journée 10 terminée, 614 id uniques cumulés jusqu'ici
Journée 11 terminée, 617 id uniques cumulés jusqu'ici
Journée 12 terminée, 621 id uniques cumulés jusqu'ici
Journée 13 terminée, 628 id uniques cumulés jusqu'ici
Journée 14 terminée, 628 id uniques cumulés jusqu'ici
Journée 15 terminée, 632 id uniques cumulés jusqu'ici
Journée 16 terminée, 650 id uniques cumulés jusqu'ici
Journée 17 terminée, 657 id uniques cumulés jusqu'ici
Journée 18 terminée, 662 id uniques cumulés jusqu'ici
Journée 19 terminée, 669 id uniques c

In [43]:
ids_deja_recuperes = set(df["id"])  # on transforme la colonne id de notre DataFrame existant en ensemble, pour comparer facilement

ids_manquants = ids_joueurs_saison - ids_deja_recuperes  # différence entre les deux ensembles : ce qui est dans les lineups mais pas encore dans notre table

print(len(ids_manquants))  # on affiche le nombre d'id à requêter en plus

385


In [44]:
id_test = list(ids_manquants)[0]  # on prend un id au hasard parmi les 385 manquants, pour tester

url_player_info = f"https://api.sofascore.com/api/v1/player/{id_test}"  # endpoint profil joueur, censé donner nom, poste, date de naissance

response_player_info = scraper.get(url_player_info)  # requête avec le scraper

print(response_player_info.status_code)  # on vérifie que ça passe

data_player_info = response_player_info.json()  # transforme en dictionnaire Python

print(data_player_info.keys())  # on regarde la structure avant de creuser dedans

200
dict_keys(['player'])


In [45]:
print(data_player_info["player"].keys())  # on liste les clés disponibles, pour repérer name, position, dateOfBirth et team

print(data_player_info["player"]["name"], "-", data_player_info["player"].get("position"), "-", data_player_info["player"].get("team", {}).get("name"))  # affiche un résumé lisible pour vérifier que les infos sont cohérentes

dict_keys(['name', 'firstName', 'lastName', 'slug', 'shortName', 'team', 'position', 'positionsDetailed', 'jerseyNumber', 'height', 'dateOfBirth', 'preferredFoot', 'userCount', 'deceased', 'gender', 'sofascoreId', 'country', 'id', 'underage', 'shirtNumber', 'dateOfBirthTimestamp', 'contractUntilTimestamp', 'proposedMarketValue', 'proposedMarketValueRaw', 'fieldTranslations'])
Simone Scaglia - G - Padova


In [46]:
url_stats_test = f"https://api.sofascore.com/api/v1/player/{id_test}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # même endpoint stats qu'on utilise depuis le début

response_stats_test = scraper.get(url_stats_test)  # requête avec le scraper

print(response_stats_test.status_code)  # on vérifie que ça passe

data_stats_test = response_stats_test.json()  # transforme en dictionnaire Python

print(data_stats_test.get("team", {}).get("name"))  # on vérifie si l'équipe donnée ici correspond bien à la saison 25/26 plutôt qu'à l'équipe actuelle

404
None


In [48]:
for id_joueur in ids_manquants:  # on parcourt les 385 id identifiés comme manquants dans notre table actuelle
    url_stats = f"https://api.sofascore.com/api/v1/player/{id_joueur}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # stats individuelles pour la saison 25/26

    response_stats = scraper.get(url_stats)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes, comme pour toutes les requêtes précédentes

    if response_stats.status_code != 200:  # cas attendu : pas de stats car le joueur n'a jamais joué de minutes (sous le seuil de 900 de toute façon)
        continue  # id suivant, pas besoin de logger vu que c'est un cas normal et attendu

    data_stats = response_stats.json()  # transforme la réponse en dictionnaire Python

    url_player_info = f"https://api.sofascore.com/api/v1/player/{id_joueur}"  # infos d'identité (nom, poste, date de naissance) pour ce joueur
    response_player_info = scraper.get(url_player_info)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes pour cette requête aussi

    if response_player_info.status_code != 200:  # sécurité : peu probable si les stats existent, mais on vérifie quand même
        continue  # id suivant

    data_player_info = response_player_info.json()["player"]  # transforme et extrait directement le sous-dictionnaire "player"

    player_info_complet = {
        "id": data_player_info["id"],  # id sofascore du joueur
        "name": data_player_info["name"],  # nom complet
        "position": data_player_info["position"],  # poste en une lettre
        "dateOfBirth": data_player_info.get("dateOfBirth"),  # date de naissance
        "team": data_stats.get("team", {}).get("name"),  # équipe prise depuis les stats de la saison 25/26, pas depuis le profil joueur (qui donne l'équipe actuelle, potentiellement fausse comme on vient de le voir)
    }  # on reconstruit un dictionnaire au même format que "player_info" utilisé dans notre fonction extraire_stats_joueur

    ligne = extraire_stats_joueur(player_info_complet, data_stats)  # même fonction d'extraction que pour les 387 premiers joueurs

    toutes_les_lignes.append(ligne)  # on ajoute directement à la liste existante, pas besoin d'une nouvelle liste séparée

print(f"Total après complétion : {len(toutes_les_lignes)} joueurs")  # bilan final, devrait être proche de 387 + (385 - nombre de 404)

TypeError: string indices must be integers, not 'str'

In [49]:
    player_info_complet = {
        "id": data_player_info["id"],  # id sofascore du joueur
        "name": data_player_info["name"],  # nom complet
        "position": data_player_info["position"],  # poste en une lettre
        "dateOfBirth": data_player_info.get("dateOfBirth"),  # date de naissance
        "team": {"name": data_stats.get("team", {}).get("name")},  # on remet team sous forme de dictionnaire {"name": ...}, comme dans le roster, pour que extraire_stats_joueur fonctionne sans modification
    }

In [50]:
for id_joueur in ids_manquants:  # on parcourt les 385 id identifiés comme manquants dans notre table actuelle
    url_stats = f"https://api.sofascore.com/api/v1/player/{id_joueur}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # stats individuelles pour la saison 25/26

    response_stats = scraper.get(url_stats)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes, comme pour toutes les requêtes précédentes

    if response_stats.status_code != 200:  # cas attendu : pas de stats car le joueur n'a jamais joué de minutes (sous le seuil de 900 de toute façon)
        continue  # id suivant, pas besoin de logger vu que c'est un cas normal et attendu

    data_stats = response_stats.json()  # transforme la réponse en dictionnaire Python

    url_player_info = f"https://api.sofascore.com/api/v1/player/{id_joueur}"  # infos d'identité (nom, poste, date de naissance) pour ce joueur
    response_player_info = scraper.get(url_player_info)  # requête avec le scraper
    time.sleep(2)  # pause de 2 secondes pour cette requête aussi

    if response_player_info.status_code != 200:  # sécurité : peu probable si les stats existent, mais on vérifie quand même
        continue  # id suivant

    data_player_info = response_player_info.json()["player"]  # transforme et extrait directement le sous-dictionnaire "player"

    player_info_complet = {
        "id": data_player_info["id"],  # id sofascore du joueur
        "name": data_player_info["name"],  # nom complet
        "position": data_player_info["position"],  # poste en une lettre
        "dateOfBirth": data_player_info.get("dateOfBirth"),  # date de naissance
        "team": {"name": data_stats.get("team", {}).get("name")},  # équipe sous forme de dictionnaire {"name": ...}, prise depuis les stats 25/26 (pas depuis le profil joueur, qui donnerait l'équipe actuelle potentiellement fausse) ; format aligné avec ce qu'attend extraire_stats_joueur
    }  # on reconstruit un dictionnaire au même format que "player_info" utilisé dans notre fonction extraire_stats_joueur

    ligne = extraire_stats_joueur(player_info_complet, data_stats)  # même fonction d'extraction que pour les 387 premiers joueurs

    toutes_les_lignes.append(ligne)  # on ajoute directement à la liste existante, pas besoin d'une nouvelle liste séparée

print(f"Total après complétion : {len(toutes_les_lignes)} joueurs")  # bilan final, devrait être proche de 387 + (385 - nombre de 404/erreurs)

Total après complétion : 586 joueurs


In [51]:
df = pd.DataFrame(toutes_les_lignes)  # on reconstruit le DataFrame à partir de la liste complète (387 initiaux + complétion, avec doublons potentiels)

df = df.drop_duplicates(subset="id", keep="last")  # on supprime les doublons sur l'id, en gardant la dernière occurrence (la plus récente, issue du deuxième passage)

print(df.shape)  # on vérifie le nombre final de lignes et de colonnes

(586, 37)


In [52]:
print(df["minutesPlayed"].isna().sum())  # on vérifie d'abord s'il y a des valeurs manquantes sur les minutes jouées, avant de filtrer (une valeur manquante ne doit pas être confondue avec 0 minute)

df_filtre = df[df["minutesPlayed"] >= 900].copy()  # on ne garde que les joueurs ayant joué au moins 900 minutes sur la saison, le .copy() évite un avertissement pandas lors de modifications futures

print(df_filtre.shape)  # on vérifie combien de joueurs restent après ce filtre

0
(339, 37)


In [53]:
df_filtre.to_csv("serie_a_2526_sofascore.csv", index=False)  # sauvegarde en CSV sur ton disque, pour ne pas avoir à tout rejouer si le notebook redémarre

In [1]:
import pandas as pd  # pour recharger et manipuler le DataFrame

df_filtre = pd.read_csv("serie_a_2526_sofascore.csv")  # on recharge directement le résultat déjà filtré et sauvegardé hier, pas besoin de tout re-scraper

print(df_filtre.shape)  # vérification que ça correspond bien à ce qu'on avait (339, 37)

(339, 37)


In [32]:
print(df_filtre["position"].value_counts())  # on affiche toutes les valeurs uniques de la colonne position, avec leur nombre d'occurrences, pour voir exactement ce qu'il faut regrouper

position
M    135
D    105
F     75
G     24
Name: count, dtype: int64


In [34]:
mapping_postes = {
    "G": "Gardien",  # catégorie à part, stats trop différentes des autres postes pour être comparées ensemble
    "D": "Défenseur",
    "M": "Milieu",
    "F": "Attaquant",
}  # dictionnaire de correspondance entre le code Sofascore et le regroupement large qu'on veut utiliser

df_filtre["poste_large"] = df_filtre["position"].map(mapping_postes)  # on crée une nouvelle colonne avec le regroupement, sans toucher à la colonne "position" d'origine

print(df_filtre["poste_large"].value_counts())  # on vérifie que le mapping a bien fonctionné et que personne n'est resté sans catégorie (NaN)

poste_large
Milieu       135
Défenseur    105
Attaquant     75
Gardien       24
Name: count, dtype: int64


In [35]:
colonnes_pourcentage = [
    "accuratePassesPercentage",  # déjà un taux, pas besoin de conversion
    "aerialDuelsWonPercentage",  # déjà un taux
]  # colonnes qu'on laisse intactes, ramenées à 90 minutes elles perdraient leur sens

colonnes_a_exclure_du_per90 = (
    ["id", "name", "position", "dateOfBirth", "team", "poste_large"]  # colonnes d'identité, pas des stats
    + ["minutesPlayed", "appearances", "matchesStarted"]  # volume de temps de jeu lui-même, sert de dénominateur, pas à convertir
    + colonnes_pourcentage  # les taux déjà exclus juste au-dessus
)  # tout ce qui ne doit PAS être transformé en "par 90"

colonnes_a_convertir = [col for col in df_filtre.columns if col not in colonnes_a_exclure_du_per90]  # toutes les colonnes restantes sont des volumes à convertir (buts, xG, tacles, passes, etc.)

print(colonnes_a_convertir)  # on vérifie la liste avant de lancer la conversion, pour être sûr de ne rien avoir oublié ou inclus par erreur

['goals', 'expectedGoals', 'assists', 'expectedAssists', 'bigChancesCreated', 'totalShots', 'shotsOnTarget', 'successfulDribbles', 'keyPasses', 'accuratePasses', 'totalPasses', 'accurateLongBalls', 'touches', 'tackles', 'tacklesWon', 'interceptions', 'clearances', 'blockedShots', 'aerialDuelsWon', 'groundDuelsWon', 'yellowCards', 'redCards', 'fouls', 'saves', 'cleanSheet', 'goalsConceded', 'penaltySave', 'nom_normalise', 'match_nom_understat', 'match_score']


In [36]:
df_filtre = df_filtre.drop(columns=["nom_normalise", "match_nom_understat", "match_score"], errors="ignore")  # on enlève les colonnes restes d'Understat ; errors="ignore" évite un crash si l'une d'elles n'existe pas (ex: si tu as rechargé le CSV, qui ne les contient pas)

colonnes_a_convertir = [col for col in df_filtre.columns if col not in colonnes_a_exclure_du_per90]  # on refait la liste maintenant que les colonnes parasites sont parties

print(colonnes_a_convertir)  # vérification finale avant de lancer la conversion

['goals', 'expectedGoals', 'assists', 'expectedAssists', 'bigChancesCreated', 'totalShots', 'shotsOnTarget', 'successfulDribbles', 'keyPasses', 'accuratePasses', 'totalPasses', 'accurateLongBalls', 'touches', 'tackles', 'tacklesWon', 'interceptions', 'clearances', 'blockedShots', 'aerialDuelsWon', 'groundDuelsWon', 'yellowCards', 'redCards', 'fouls', 'saves', 'cleanSheet', 'goalsConceded', 'penaltySave']


In [37]:
for colonne in colonnes_a_convertir:  # on parcourt chaque colonne de volume identifiée
    df_filtre[colonne + "_per90"] = df_filtre[colonne] / df_filtre["minutesPlayed"] * 90  # règle de trois : (valeur / minutes jouées) donne un taux par minute, qu'on multiplie par 90 pour ramener à un match complet

print(df_filtre[["name", "goals", "minutesPlayed", "goals_per90"]].head())  # on vérifie sur un exemple simple (buts) que le calcul est cohérent

               name  goals  minutesPlayed  goals_per90
0  Lautaro Martínez     17           2178     0.702479
1     Marcus Thuram     13           1913     0.611605
2   Ange-Yoan Bonny      5           1155     0.389610
3      Pio Esposito      7           1572     0.400763
4  Hakan Çalhanoğlu      9           1649     0.491207


In [38]:
colonnes_attaque = [
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",
    "totalShots_per90", "shotsOnTarget_per90", "bigChancesCreated_per90",
    "successfulDribbles_per90", "keyPasses_per90", "touches_per90",
]

colonnes_milieu = [
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",
    "keyPasses_per90", "accuratePasses_per90", "accuratePassesPercentage",
    "accurateLongBalls_per90", "successfulDribbles_per90", "touches_per90",
    "tackles_per90", "interceptions_per90",
    "groundDuelsWon_per90", "aerialDuelsWon_per90",  # ajoutés, un milieu doit aussi se battre dans les duels pour récupérer/protéger le ballon
]

colonnes_defense = [
    "tackles_per90", "tacklesWon_per90", "interceptions_per90", "clearances_per90",
    "blockedShots_per90", "aerialDuelsWon_per90", "aerialDuelsWonPercentage",
    "groundDuelsWon_per90", "accuratePasses_per90", "accuratePassesPercentage",
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",  # ajoutés, pour distinguer un défenseur offensif d'un pur défenseur
    "successfulDribbles_per90", "keyPasses_per90", "touches_per90",  # ajoutés, pour juger l'apport technique/relance
]

colonnes_gardien = [
    "saves_per90", "goalsConceded_per90", "penaltySave_per90",
    "cleanSheet",  # volume brut, pas per90, un clean sheet est un événement par match entier et non un taux continu
]

In [39]:
from sklearn.preprocessing import StandardScaler  # outil qui centre-réduit chaque colonne (moyenne 0, écart-type 1)

def normaliser_groupe(df, colonnes):
    # df : sous-ensemble du DataFrame pour un seul poste (ex: uniquement les défenseurs)
    # colonnes : liste des colonnes de stats à normaliser pour ce poste précis
    scaler = StandardScaler()  # un nouveau scaler à chaque appel, pour ne pas mélanger les échelles entre postes différents
    valeurs_normalisees = scaler.fit_transform(df[colonnes])  # calcule et applique la normalisation en une fois sur ce sous-groupe
    df_normalise = pd.DataFrame(valeurs_normalisees, columns=[c + "_norm" for c in colonnes], index=df.index)  # on garde le même index que df, essentiel pour pouvoir recoller les colonnes ensuite sans décalage
    return pd.concat([df, df_normalise], axis=1)  # on colle les colonnes normalisées à côté des colonnes d'origine, sans les remplacer

df_attaque = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Attaquant"], colonnes_attaque)  # sous-table attaquants avec leurs colonnes normalisées
df_milieu = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Milieu"], colonnes_milieu)  # sous-table milieux
df_defense = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Défenseur"], colonnes_defense)  # sous-table défenseurs
df_gardien = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Gardien"], colonnes_gardien)  # sous-table gardiens

print(df_attaque[[c + "_norm" for c in colonnes_attaque]].describe().loc[["mean", "std"]])  # vérification : la moyenne doit être proche de 0 et l'écart-type proche de 1 pour chaque colonne normalisée

      goals_per90_norm  expectedGoals_per90_norm  assists_per90_norm  \
mean      3.444467e-16             -5.477100e-17        1.672736e-16   
std       1.006734e+00              1.006734e+00        1.006734e+00   

      expectedAssists_per90_norm  totalShots_per90_norm  \
mean               -2.279658e-16          -3.274233e-16   
std                 1.006734e+00           1.006734e+00   

      shotsOnTarget_per90_norm  bigChancesCreated_per90_norm  \
mean              1.403276e-16                  1.184238e-17   
std               1.006734e+00                  1.006734e+00   

      successfulDribbles_per90_norm  keyPasses_per90_norm  touches_per90_norm  
mean                   7.080563e-18          5.343873e-16        7.579123e-16  
std                    1.006734e+00          1.006734e+00        1.006734e+00  


In [40]:
print(df_attaque[[c + "_norm" for c in colonnes_attaque]].describe().loc[["mean", "std"]].round(3))  # arrondi à 3 décimales pour une lecture plus claire

      goals_per90_norm  expectedGoals_per90_norm  assists_per90_norm  \
mean             0.000                    -0.000               0.000   
std              1.007                     1.007               1.007   

      expectedAssists_per90_norm  totalShots_per90_norm  \
mean                      -0.000                 -0.000   
std                        1.007                  1.007   

      shotsOnTarget_per90_norm  bigChancesCreated_per90_norm  \
mean                     0.000                         0.000   
std                      1.007                         1.007   

      successfulDribbles_per90_norm  keyPasses_per90_norm  touches_per90_norm  
mean                          0.000                 0.000               0.000  
std                           1.007                 1.007               1.007  


In [41]:
from sklearn.metrics.pairwise import cosine_similarity  # calcule la similarité cosinus entre toutes les paires de lignes d'une matrice, en une seule fois

def calculer_similarite(df, colonnes):
    # df : sous-table d'un seul poste, déjà normalisée
    # colonnes : liste des colonnes de base (sans le suffixe _norm), utilisées pour retrouver les bonnes colonnes normalisées
    colonnes_norm = [c + "_norm" for c in colonnes]  # on reconstruit les noms des colonnes normalisées à utiliser
    matrice_similarite = cosine_similarity(df[colonnes_norm])  # calcule un score de similarité (entre -1 et 1) entre chaque paire de joueurs de ce sous-groupe
    df_similarite = pd.DataFrame(matrice_similarite, index=df["name"], columns=df["name"])  # on transforme en DataFrame lisible, avec les noms des joueurs en lignes ET en colonnes
    return df_similarite

similarite_attaque = calculer_similarite(df_attaque, colonnes_attaque)  # matrice de similarité entre tous les attaquants

print(similarite_attaque["Lautaro Martínez"].sort_values(ascending=False).head(6))  # on regarde les 6 joueurs les plus proches de Lautaro Martínez (le premier sera lui-même à 1.0)

name
Lautaro Martínez    1.000000
Marcus Thuram       0.953882
Nikola Krstović     0.857244
Rafael Leão         0.831344
Donyell Malen       0.785491
Dušan Vlahović      0.749325
Name: Lautaro Martínez, dtype: float64


In [42]:
print(similarite_attaque["Rafael Leão"].sort_values(ascending=False).head(6))  # les 6 attaquants les plus proches de Leão, pour voir si le profil "ailier technique" ressort différemment du profil "numéro 9" de Lautaro

name
Rafael Leão          1.000000
Lautaro Martínez     0.831344
Christian Pulišić    0.770941
Nikola Krstović      0.713712
Marcus Thuram        0.702615
Donyell Malen        0.683487
Name: Rafael Leão, dtype: float64


In [43]:
print(df_attaque[df_attaque["name"].isin(["Rafael Leão", "Lautaro Martínez"])][["name"] + colonnes_attaque].set_index("name").T)  # on affiche les valeurs brutes (pas normalisées) des deux joueurs côte à côte, colonne par colonne, pour voir où elles se ressemblent et où elles divergent

name                      Lautaro Martínez  Rafael Leão
goals_per90                       0.702479     0.435718
expectedGoals_per90               0.557669     0.478143
assists_per90                     0.247934     0.145239
expectedAssists_per90             0.111909     0.163224
totalShots_per90                  3.801653     3.050027
shotsOnTarget_per90               1.611570     1.161915
bigChancesCreated_per90           0.495868     0.484131
successfulDribbles_per90          0.867769     1.258741
keyPasses_per90                   1.528926     1.113502
touches_per90                    42.314050    44.104357


In [44]:
similarite_milieu = calculer_similarite(df_milieu, colonnes_milieu)  # matrice de similarité pour tous les milieux
similarite_defense = calculer_similarite(df_defense, colonnes_defense)  # matrice de similarité pour tous les défenseurs
similarite_gardien = calculer_similarite(df_gardien, colonnes_gardien)  # matrice de similarité pour tous les gardiens

print(similarite_milieu.shape, similarite_defense.shape, similarite_gardien.shape)  # vérification rapide que chaque matrice a bien une taille carrée correspondant au nombre de joueurs du poste (135, 105, 24)

(135, 135) (105, 105) (24, 24)


In [45]:
def convertir_en_pourcentage(df_similarite):
    # df_similarite : matrice de similarité cosinus (valeurs entre -1 et 1)
    return ((df_similarite + 1) / 2 * 100).round(1)  # ramène l'intervalle [-1, 1] vers [0, 100], arrondi à 1 décimale pour l'affichage

pourcentage_attaque = convertir_en_pourcentage(similarite_attaque)  # conversion pour chaque poste
pourcentage_milieu = convertir_en_pourcentage(similarite_milieu)
pourcentage_defense = convertir_en_pourcentage(similarite_defense)
pourcentage_gardien = convertir_en_pourcentage(similarite_gardien)

print(pourcentage_attaque["Lautaro Martínez"].sort_values(ascending=False).head(6))  # on revérifie sur Lautaro, cette fois en pourcentage, pour confirmer que la conversion garde le même classement qu'avant

name
Lautaro Martínez    100.0
Marcus Thuram        97.7
Nikola Krstović      92.9
Rafael Leão          91.6
Donyell Malen        89.3
Dušan Vlahović       87.5
Name: Lautaro Martínez, dtype: float64


In [46]:
def chercher_joueurs_similaires(nom_joueur, top_n=10):
    # nom_joueur : nom exact du joueur recherché, tel qu'il apparaît dans la colonne "name"
    # top_n : nombre de joueurs similaires à retourner, 10 par défaut

    ligne_joueur = df_filtre[df_filtre["name"] == nom_joueur]  # on cherche la ligne correspondant au joueur demandé

    if ligne_joueur.empty:  # sécurité : si le nom n'existe pas exactement dans la table, on prévient plutôt que de planter
        print(f"Joueur '{nom_joueur}' introuvable, vérifie l'orthographe exacte")
        return None

    poste = ligne_joueur["poste_large"].iloc[0]  # on récupère le poste large du joueur, pour savoir quelle matrice et quelle table utiliser

    # on choisit la bonne matrice de pourcentage et la bonne table de stats selon le poste, via un dictionnaire plutôt qu'une longue série de if/else
    matrices_par_poste = {
        "Attaquant": (pourcentage_attaque, df_attaque, colonnes_attaque),
        "Milieu": (pourcentage_milieu, df_milieu, colonnes_milieu),
        "Défenseur": (pourcentage_defense, df_defense, colonnes_defense),
        "Gardien": (pourcentage_gardien, df_gardien, colonnes_gardien),
    }

    matrice_pourcentage, df_poste, colonnes_stats = matrices_par_poste[poste]  # on récupère les 3 éléments correspondant au poste du joueur

    joueurs_proches = matrice_pourcentage[nom_joueur].sort_values(ascending=False).iloc[1:top_n+1]  # on prend les scores triés, en excluant iloc[0] qui est le joueur lui-même à 100%

    resultat = df_poste[df_poste["name"].isin(joueurs_proches.index)][["name", "team", "position"] + colonnes_stats].copy()  # on récupère les stats des joueurs similaires, identité + colonnes pertinentes pour ce poste

    resultat["pourcentage_ressemblance"] = resultat["name"].map(joueurs_proches)  # on ajoute le pourcentage de ressemblance en dernière colonne, en le faisant correspondre au bon nom

    resultat = resultat.sort_values("pourcentage_ressemblance", ascending=False).reset_index(drop=True)  # tri final par ressemblance décroissante, index remis à zéro pour un affichage propre

    return resultat

resultat_test = chercher_joueurs_similaires("Lautaro Martínez", top_n=5)  # test sur Lautaro, comme dans nos essais précédents
print(resultat_test)

              name      team position  goals_per90  expectedGoals_per90  \
0    Marcus Thuram     Inter        F     0.611605             0.443832   
1  Nikola Krstović  Atalanta        F     0.503919             0.706772   
2      Rafael Leão  AC Milan        F     0.435718             0.478143   
3    Donyell Malen   AS Roma        F     0.852503             0.863184   
4   Dušan Vlahović  Juventus        F     0.645492             0.652850   

   assists_per90  expectedAssists_per90  totalShots_per90  \
0       0.282279               0.120354          3.481443   
1       0.251960               0.106895          4.988802   
2       0.145239               0.163224          3.050027   
3       0.121786               0.145694          4.262517   
4       0.092213               0.110040          4.518443   

   shotsOnTarget_per90  bigChancesCreated_per90  successfulDribbles_per90  \
0             1.364349                 0.329326                  0.799791   
1             1.713326      

In [48]:
!pip install sqlalchemy pymysql

In [50]:
utilisateur = "root"
mot_de_passe = ""
hote = "localhost"
port = 3307  # port corrigé, celui affiché dans le panneau de contrôle XAMPP pour MySQL
nom_base = "recruefoot"

url_connexion = f"mysql+pymysql://{utilisateur}:{mot_de_passe}@{hote}:{port}/{nom_base}"

engine = create_engine(url_connexion)

with engine.connect() as connexion:
    print("Connexion réussie !")

Connexion réussie !


In [52]:
def matrice_vers_table_longue(matrice_pourcentage, poste):
    # matrice_pourcentage : DataFrame carré (joueur x joueur) avec les scores de ressemblance
    # poste : nom du poste concerné, pour l'ajouter comme colonne et garder la trace du groupe d'origine
    matrice_renommee = matrice_pourcentage.rename_axis(index="joueur", columns="joueur_compare")  # on donne des noms différents à l'index et aux colonnes, pour éviter le conflit lors du reset_index()
    table_longue = matrice_renommee.stack().reset_index()  # transforme la matrice carrée en 3 colonnes : joueur, joueur_compare, valeur
    table_longue.columns = ["joueur", "joueur_compare", "pourcentage_ressemblance"]  # on renomme la 3e colonne générée par stack(), qui n'a pas de nom par défaut
    table_longue = table_longue[table_longue["joueur"] != table_longue["joueur_compare"]]  # on retire les lignes où un joueur est comparé à lui-même
    table_longue["poste"] = poste  # on ajoute le poste

    return table_longue

table_similarites = pd.concat([  # on empile les 4 tables longues (une par poste) en une seule table finale
    matrice_vers_table_longue(pourcentage_attaque, "Attaquant"),
    matrice_vers_table_longue(pourcentage_milieu, "Milieu"),
    matrice_vers_table_longue(pourcentage_defense, "Défenseur"),
    matrice_vers_table_longue(pourcentage_gardien, "Gardien"),
], ignore_index=True)

print(table_similarites.shape)  # vérification de la taille
print(table_similarites.head())  # aperçu du résultat

(35112, 4)
             joueur   joueur_compare  pourcentage_ressemblance      poste
0  Lautaro Martínez    Marcus Thuram                      97.7  Attaquant
1  Lautaro Martínez  Ange-Yoan Bonny                      65.8  Attaquant
2  Lautaro Martínez     Pio Esposito                      82.5  Attaquant
3  Lautaro Martínez   Rasmus Højlund                      56.1  Attaquant
4  Lautaro Martínez      David Neres                      54.8  Attaquant


In [53]:
df_filtre.to_sql("joueurs", con=engine, if_exists="replace", index=False)  # exporte df_filtre vers une table "joueurs" ; if_exists="replace" écrase la table si elle existe déjà (utile si tu relances ce code plus tard après une mise à jour des données)

table_similarites.to_sql("similarites", con=engine, if_exists="replace", index=False)  # exporte la table de ressemblance vers une table "similarites"

print("Export terminé")

Export terminé
